In [1]:
import pandas as pd 
import os 
import sys
from dotenv import load_dotenv
from pathlib import Path 
import re 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

notebook_path = Path.cwd()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# from src.data import load_db
#This extension will reload this cell every time that the code is executed (it helps with loading modules imported from local files)

pd.set_option('display.max_rows', 90)
%load_ext autoreload
%autoreload 2

# Dochód jako kluczowy predyktor w modelu Churn

Dochód jest jednym z najsilniejszych predyktorów demograficznych w modelu churn, ponieważ bezpośrednio wpływa na wrażliwość cenową klienta oraz jego całkowitą wartość cyklu życia (LTV - Lifetime Value).

Integrując dane o dochodach na poziomie kodów pocztowych (ZIP-code) dla Kalifornii – pochodzące z IRS (urzędu skarbowego) oraz Census Bureau (biura spisu ludności) – możemy wyjść poza proste flagi demograficzne. Pozwala nam to zrozumieć, w jaki sposób lokalne środowisko ekonomiczne wpływa na decyzję klienta o pozostaniu lub rezygnacji z usług.

W tym pliku zrealizuję **strukturyzację danych**, aby przygotować je do połączenia i załadowania do mojej bazy **PostgreSQL**.

In [2]:
cdf = pd.read_excel('../data/raw_data/Census_income_data.xlsx', sheet_name='Data')

In [3]:
cdf.head()

,Unnamed: 0,ZCTA5 89010,Unnamed: 2,Unnamed: 3,Unnamed: 4,ZCTA5 89019,Unnamed: 6,Unnamed: 7,Unnamed: 8,ZCTA5 89060,...,Unnamed: 7223,Unnamed: 7224,ZCTA5 96161,Unnamed: 7226,Unnamed: 7227,Unnamed: 7228,ZCTA5 97635,Unnamed: 7230,Unnamed: 7231,Unnamed: 7232
0,NaN,Households,Families,Married-couple families,Nonfamily households,Households,Families,Married-couple families,Nonfamily households,Households,...,Married-couple families,Nonfamily households,Households,Families,Married-couple families,Nonfamily households,Households,Families,Married-couple families,Nonfamily households
1,Label,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,...,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate,Estimate
2,Total,165,109,96,56,"1,198",623,535,575,"5,163",...,0,0,"7,655","5,195","4,692","2,460",107,97,87,10
3,"Less than $10,000",0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,5.1%,...,-,-,5.2%,2.2%,2.2%,12.0%,9.3%,10.3%,0.0%,0.0%
4,"$10,000 to $14,999",0.0%,0.0%,0.0%,0.0%,3.3%,0.0%,0.0%,7.0%,4.2%,...,-,-,1.8%,0.3%,0.0%,4.7%,0.0%,0.0%,0.0%,0.0%


In [4]:
valid_cols = [col for col in cdf.columns if 'Unnamed' not in col]

In [5]:
cdf_t = cdf.loc[[13,14],valid_cols].T
cdf_t = cdf_t.reset_index()
cdf_t = cdf_t.rename(columns={'index':'zip_code',
                      13: 'median_income',
                      14: 'mean_income'})

#cleaning the zip codes eg. removing the prefix

cdf_t['zip_code'] = cdf_t['zip_code'].apply(lambda x: x.split(' ')[1])
cdf_t['mean_income'] = cdf_t['mean_income'].str.replace(',','')
cdf_t['median_income'] = cdf_t['median_income'].str.replace(',','')

In [6]:
cdf_t['mean_income'] = cdf_t['mean_income'].apply(lambda x: None if x == '-' else str(x).replace(',',''))
cdf_t['median_income'] =  cdf_t['median_income'].apply(lambda x: None if x == '-' else str(x).replace(',',''))

In [7]:
cdf_t['mean_income'] = pd.to_numeric(cdf_t['mean_income'], errors='coerce')
cdf_t['median_income'] = pd.to_numeric(cdf_t['mean_income'], errors='coerce')

In [8]:
cdf_t.sort_values(by='zip_code')

,zip_code,median_income,mean_income
0,89010,68153.0,68153.0
1,89019,70883.0,70883.0
2,89060,75975.0,75975.0
3,89061,87289.0,87289.0
4,89439,142472.0,142472.0
...,...,...,...
1803,96148,94647.0,94647.0
1804,96150,128072.0,128072.0
1805,96155,NaN,NaN
1806,96161,190864.0,190864.0


In [9]:
cdf_t['median_income'].iloc[66]

np.float64(nan)

In [10]:
cdf_t.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1808 entries, 0 to 1807
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   zip_code       1808 non-null   object 
 1   median_income  1679 non-null   float64
 2   mean_income    1679 non-null   float64
dtypes: float64(2), object(1)
memory usage: 42.5+ KB
